# 第二章（四）：字符串与文本处理

**授课教师：** 王润祖

In [ ]:
import os

os.makedirs("data", exist_ok=True)
with open("./data/data.txt", "w", encoding="utf-8") as f:
    f.write("name,gender,income\n Alice , FEMALE , ¥1200 \nBob, male ,N/A\n Cathy, Female ,5000\n")

## 本节概览

* 理解文本文件读进程序后通常仍然是字符串
* 掌握 `read()`、`readlines()`、`strip()`、`split()`、`replace()`、`lower()` 的基础用法
* 学会把文本清洗步骤写成函数
* 能够用循环和异常处理完成简单脏数据清洗

# 读取文本文件

## 一个最基本的读取框架

* 在统计学习中，原始数据往往不是直接写在代码里，而是放在文件中。
* `with open(...)` 是 Python 中最常见的文件读取框架。

```python
with open("./data/data.txt", "r", encoding="utf-8") as f:
    content = f.read()
```

## `read()`

* `read()` 会把整个文件内容一次读进来。
* 读进来的结果通常是一个大字符串。
* 也就是说，文件能读进来，不等于数据已经能直接分析。

In [ ]:
with open("./data/data.txt", "r", encoding="utf-8") as f:
    text = f.read()

print(text)
print(type(text))

## `readlines()`

* `readlines()` 会把文件按行读入列表。
* 列表中的每个元素，本质上仍然是字符串。

In [ ]:
with open("./data/data.txt", "r", encoding="utf-8") as f:
    lines = f.readlines()

print(lines)
print(type(lines))

# 读进来了，但还不能直接用

## 一行文本数据

* 下面这一行来自文件，看起来像“有姓名、有性别、有收入”。
* 但对程序来说，它还只是文本。
* 真正的数据分析工作，往往就从“把这行文本变干净”开始。

In [ ]:
print(lines[1])

## 现在的问题

* 有多余空格
* 不同字段还挤在一起
* 数字还是文本

# 字符串的基础处理

## 去空白：`strip()`

* 文本数据里常常混有空格和换行符。
* `strip()` 可以先把最外层多余空白去掉。

In [ ]:
line = " Alice , FEMALE , ¥1200 \n"
print(line.strip())

## 按分隔符拆开：`split()`

* 当一行数据里有多个字段时，通常要先拆开。
* `split(",")` 会把一行文本拆成若干部分。

In [ ]:
line = "Alice , FEMALE ,¥1200"
parts = line.split(",")
print(parts)
print(parts[0])
print(parts[1])
print(parts[2])

## 替换字符：`replace()`

* 有些字符只是“格式符号”，不是我们真正想分析的数据。
* 例如货币符号和千分位逗号，往往需要先去掉。

In [ ]:
text = "¥1,200"
text = text.replace("¥", "")
text = text.replace(",", "")
print(text)

## 统一大小写：`lower()`

* 分类变量常常会出现大小写不一致的问题。
* `lower()` 可以把它们统一成小写。
* 如果不先统一，程序会把同一类数据误当成不同类别。

In [ ]:
gender = " FEMALE "
gender = gender.strip().lower()
print(gender)

## 文本转数值

* 现在看起来像数字的内容，仍然只是字符串。
* 要参与计算，必须先转成真正的数值类型。

In [ ]:
income_text = "1200"
income = int(income_text)
print(income)
print(type(income))

## 小结任务：课堂任务 1

给定一行问卷文本，补全下面的代码：去掉首尾空白、拆成三个字段，并输出清理后的姓名、统一成小写的性别，以及去掉货币符号后的收入文本。

In [ ]:
line = " Alice , Female , ¥1200 \n"
clean_line = # 在这里补全
parts = # 在这里补全

print(parts[0].strip())
print(parts[1].strip().lower())
print(parts[2].strip().replace("¥", ""))

# 统一格式并封装清洗规则

## 同一类别可能有不同写法

* 同一类别如果大小写和空格不一致，程序会把它们当成不同的值。
* 清洗规则应统一、明确，并在重复使用时封装成函数。

In [ ]:
raw_genders = ["Male", " male ", "FEMALE", " female "]
clean_genders = [gender.strip().lower() for gender in raw_genders]
print(clean_genders)

## 清洗收入文本

In [ ]:
def clean_income(text):
    text = text.strip()
    text = text.replace("¥", "")
    text = text.replace(",", "")
    try:
        return float(text)
    except ValueError:
        return None

print(clean_income(" ¥1,200 "))
print(clean_income("N/A"))

## 为什么返回 `None`

* `N/A`等内容不能直接转换为数值。
* 返回`None`是在明确标记“当前没有有效数值”，不是隐瞒错误。
* 本例只捕获数值转换产生的`ValueError`，其他错误仍应暴露并检查。

## 小结任务：课堂任务2

补全`clean_percent(text)`：把百分号文本转成数值，转换失败时返回`None`。

In [ ]:
def clean_percent(text):
    text = text.strip().replace("%", "")
    try:
        # 在这里补全
        pass
    except ValueError:
        # 在这里补全
        pass

print(clean_percent(" 45% "))
print(clean_percent("N/A"))

# 综合案例：从文本行到结构化记录

## 读取文件

In [ ]:
with open("./data/data.txt", "r", encoding="utf-8") as f:
    lines = f.readlines()

print(lines)

## 逐行拆分和清洗

* 跳过第一行字段名。
* 对其余文本行依次完成拆分、清洗、转换和保存。
* 每条清洗结果使用字典表示，便于看清字段含义。

In [ ]:
clean_records = []

for line in lines[1:]:
    parts = line.strip().split(",")
    record = {
        "name": parts[0].strip(),
        "gender": parts[1].strip().lower(),
        "income": clean_income(parts[2])
    }
    clean_records.append(record)

print(clean_records)

## 检查清洗结果

* 文件能够读取，不代表数据已经正确。
* 清洗后应检查记录数、字段含义和无效值。

In [ ]:
print("记录数:", len(clean_records))

for record in clean_records:
    if record["income"] is None:
        print("收入无效:", record["name"])

## 本例回顾

* 使用`with open(...)`和`readlines()`读取文本文件；
* 使用字符串方法拆分并统一字段；
* 使用函数和`ValueError`处理数值转换；
* 使用循环、字典和列表保存结构化结果。

### 先掌握主线
本节先掌握“读取—拆分—清洗—检查”的完整流程，再逐步处理更长、更复杂的清洗任务。

# 课堂练习

## 练习1

给定字符串 `" Bob , Male , ¥980 \n"`，补全代码：清理后拆成三个字段，并输出姓名、统一成小写的性别，以及去掉货币符号后的收入文本。

In [ ]:
line = " Bob , Male , ¥980 \n"
# 提示：可以先去首尾空白、拆分字段，再分别清理性别和收入文本

## 练习2

给定列表 `["Male", " male ", "FEMALE ", " female"]`，先统一格式，再统计其中 `female` 的个数。

In [ ]:
raw_genders = ["Male", " male ", "FEMALE ", " female"]
# 提示：可以先得到 clean_genders，再统计其中有多少个 "female"

## 练习3

给定收入文本列表`["1200", "N/A", "980", " 1500 "]`，调用清洗函数，输出每个文本对应的清洗结果。

In [ ]:
raw_incomes = ["1200", "N/A", "980", " 1500 "]
# 提示：逐个调用 clean_income(text)，把结果保存到新列表

# 文本处理要有规则

* 同一种数据，格式要尽量统一。
* 缺失值、异常值、非法值，要先定义处理规则。
* 文本清洗是分析前的重要准备。

### 一点提醒
后续学习Pandas后，很多这类表格文本处理会由专门的数据处理工具更高效地完成。

# 本节小结

* 文件中的文本数据读进来后，通常还不能直接分析
* `strip()`、`split()`、`replace()`、`lower()` 是基础工具
* 可以把清洗步骤写成函数
* 可以用循环和异常处理完成一条简单清洗流水线

## 下一章

下一章将介绍 NumPy 与随机模拟，开始从“基础程序处理”过渡到“面向数据计算”。